# 03 — Modelagem

**Dimensão 5 da rúbrica — 20 pontos.**

Mínimo de **dois** classificadores distintos. Um único modelo zera 8 dos 20 pontos.

In [2]:
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns
import numpy as np

# Semente fixa: use em TODO ponto com aleatoriedade (split, modelos, CV).
RANDOM_STATE = 42

#Leitura dos arquivos processados no notebook 1 e salvos no Github
url_base_credito = f"https://github.com/camilavbr/Tech_Challenge_Fase2_rm377839/raw/main/data/processed/base_credito.csv"
url_base_credito_faixas = f"https://github.com/camilavbr/Tech_Challenge_Fase2_rm377839/raw/main/data/processed/base_credito_faixas.csv"

df = pd.read_csv(url_base_credito)
df_faixas = pd.read_csv(url_base_credito_faixas)


## 1. Split treino/teste

`stratify` preserva a proporção das classes nos dois conjuntos.

In [3]:
#Segmentação dos dados com 80% para treino e 20% para teste utilizando o stratify para manter as proporções do target no treino e no teste
from sklearn.model_selection import train_test_split

TARGET = "mau_pagador"

x = df.drop(columns=[TARGET])
y = df[TARGET]
x_train, x_test, y_train, y_test = train_test_split(
x, y, test_size=0.2, stratify=y, random_state=RANDOM_STATE
)

In [4]:
#Segmentação dos dados por faixa
from sklearn.model_selection import train_test_split

TARGET = "mau_pagador"

x_faixas = df_faixas.drop(columns=[TARGET])
y_faixas = df_faixas[TARGET]
x_faixas_train, x_faixas_test, y_faixas_train, y_faixas_test = train_test_split(
x_faixas, y_faixas, test_size=0.2, stratify=y, random_state=RANDOM_STATE
)

In [9]:
print("Colunas atuais na matriz X:")
print(x_train.columns.tolist())

Colunas atuais na matriz X:
['ID', 'GENERO', 'CARRO_PROPRIO', 'IMOVEIS', 'FILHOS', 'RENDA', 'TELEFONE_TRABALHO', 'TELEFONE', 'EMAIL', 'MEMBROS_FAMILIA', 'IDADE', 'ANOS_EMPREGADO', 'ESCOLARIDADE_NUM', 'ID.1', 'GENERO.1', 'CARRO_PROPRIO.1', 'IMOVEIS.1', 'FILHOS.1', 'RENDA.1', 'TELEFONE_TRABALHO.1', 'TELEFONE.1', 'EMAIL.1', 'MEMBROS_FAMILIA.1', 'mau_pagador.1', 'IDADE.1', 'ANOS_EMPREGADO.1', 'ESCOLARIDADE_NUM.1', 'NAME_HOUSING_TYPE_House / apartment', 'NAME_HOUSING_TYPE_Municipal apartment', 'NAME_HOUSING_TYPE_Office apartment', 'NAME_HOUSING_TYPE_Rented apartment', 'NAME_HOUSING_TYPE_With parents', 'NAME_FAMILY_STATUS_Married', 'NAME_FAMILY_STATUS_Separated', 'NAME_FAMILY_STATUS_Single / not married', 'NAME_FAMILY_STATUS_Widow', 'NAME_INCOME_TYPE_Pensioner', 'NAME_INCOME_TYPE_State servant', 'NAME_INCOME_TYPE_Student', 'NAME_INCOME_TYPE_Working']


## 2. Modelos candidatos

Usar `Pipeline` evita vazamento: o scaler é ajustado só no fold de treino durante a validação cruzada.

In [5]:
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_validate
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

modelos = {
    "logistic_regression": Pipeline([
        ("scaler", StandardScaler()),
        ("clf", LogisticRegression(max_iter=1000, random_state=RANDOM_STATE)),
    ]),
    "random_forest": Pipeline([
        ("clf", RandomForestClassifier(n_estimators=300, random_state=RANDOM_STATE, n_jobs=-1)),
    ])
}

## 3. Validação cruzada

In [7]:
from sklearn.model_selection import cross_val_score

# for nome, modelo in modelos.items():
#     scores = cross_val_score(modelo, X_train, y_train, cv=5, scoring="f1")
#     print(nome, scores.mean().round(4))

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)
resultados = []

cenarios = {
    'Base Original (Numérica Contínua)': (x_train, y_train),
    'Base de Faixas (Categórica Ordinal)': (x_faixas_train, y_faixas_train)
}

for nome_base, (x_tr, y_tr) in cenarios.items():
    for nome_modelo, modelo in modelos.items():

        # A Regressão Logística EXIGE dados padronizados na Base Original para convergir.
        # Faremos isso aqui apenas no treino usando o StandardScaler para simular o pipeline local.
        if nome_modelo == 'logistic_regression' and 'Original' in nome_base:
            scaler = StandardScaler()
            x_input = scaler.fit_transform(x_tr)
        else:
            x_input = x_tr

        # Executa a validação cruzada medindo AUC-ROC (separabilidade) e Recall (detecção de calote)
        scores = cross_validate(modelo, x_input, y_tr, cv=cv, scoring=['roc_auc', 'recall'], n_jobs=-1)

        resultados.append({
            'Abordagem de Dados': nome_base,
            'Algoritmo': nome_modelo,
            'AUC-ROC Médio': np.mean(scores['test_roc_auc']),
            'Recall Médio (Classe 1)': np.mean(scores['test_recall'])
        })

## 4. Comparação

**Leitura:** _qual modelo venceu e por qual margem? A diferença é relevante ou está dentro do ruído?_

In [8]:
df_comparativo = pd.DataFrame(resultados)
print("📊 RESULTADO DO EXPERIMENTO SOB VALIDAÇÃO CRUZADA:")
display(df_comparativo.sort_values(by='AUC-ROC Médio', ascending=False))

📊 RESULTADO DO EXPERIMENTO SOB VALIDAÇÃO CRUZADA:


,Abordagem de Dados,Algoritmo,AUC-ROC Médio,Recall Médio (Classe 1)
0,Base Original (Numérica Contínua),logistic_regression,1.0,1.0
1,Base Original (Numérica Contínua),random_forest,1.0,1.0
2,Base de Faixas (Categórica Ordinal),logistic_regression,1.0,1.0
3,Base de Faixas (Categórica Ordinal),random_forest,1.0,1.0
